## 1. Objective

We seek to understand customer behaviour, trip patterns, traffic congestions, airport demand, weather effects , and revenue trend. We began by first taking a look at the January 2026 nyc recorded yellow-trip data. Our first research item is to Understanding spatial and temporal inequalities in urban mobility using NYC taxi data. We want to know how the structure of NYC's taxi mobility network vary across neighborhoods and over time, and what does this reveal about spatial inequality in urban mobility?. The specific questions to be answered can be found in research_questions.md

## 2. Load and Inspect Data 

In [1]:
# import packages
import os  # directory management 
import pandas as pd
import pyarrow.parquet as pq #import parquet data
from pathlib import Path #import path management

 

In [2]:
# add the src folder to the path so that we can import modules from it
import sys # import sys module

#sys.path.append(os.path.join(os.path.dirname(__file__), "..", "src"))
sys.path.append("../src") # relative path to src

#print(sys.path) # print the path to check if src is added


In [4]:
# module for reading data from parquet files in src/data_ingestion.py

from data_ingestion import load_data

In [5]:


df = load_data("../data/raw/yellow_tripdata_2026-01.parquet")
print(df.head())


   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1         1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2         1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3         2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4         2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.97         1.0                  N           239           238   
1           0.90         1.0                  N           163           162   
2           1.40         1.0                  N            43           237   
3           5.58         1.0                  N           142           209   
4           2.16         1.0                  N            88           144   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [ ]:
# set directory to notebooks. set a relative path to work folder
#os.chdir('C:/Users/Denis Folitse/Desktop/data-science/project-01-urban-mobility/notebooks')
#BASE_DIR = Path('notebooks').resolve().parent
# Path to data

#path_to_data = BASE_DIR/".."/ "data" / "raw"

#ytnyc_path = path_to_data / "yellow_tripdata_2026-01.parquet"
# Check file exists
#if not ytnyc_path.exists():
#    raise FileNotFoundError(f"File not found: {ytnyc_path}")
# read in the data
#tbl = pq.read_table(ytnyc_path)
#df = tbl.to_pandas()

#print(df.head())


## 3. Data Structure

In [ ]:
# column names
print(df.columns.tolist())

['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


 Check Data structure


In [6]:
row, col = df.shape #check the number of rows and columns

print(f"\n Number of rows: {row}")
print(f"\n Number of Columns: {col}")


 Number of rows: 3724889

 Number of Columns: 20


January 2026 contains 3.72 million recorded yellow-taxi trips, providing sufficient observations for neighborhood-level aggragation.There are 20 variables it to work with.

## 4. Variable Types

In [7]:
df.info() # Checking for the column names and data type

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee            float64   

## 5. Missingness Investigation

### 5.1 Missing-value overview

In [8]:
df.isnull().sum() #check for missing values per column


VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1088058
trip_distance                  0
RatecodeID               1088058
store_and_fwd_flag       1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [10]:
#df.isna().sum().sum()

Out of these, 5 columns ( passenger count, RatecodeID, store and fwd flag, congestion sucharge, airport fee) are all missing 1,088,058 data each. The rest of the columns has a complete data. 
 

### 5.2 Missingness by vendor

### 5.3 Missingness by payment type


## 6. Negative-value investigation

### 6.1 Negative monetary values

In [11]:
# check for:

only_zeros_col = df.columns[(df==0).all()] # columns with only zeros
print("Columns with only zeros:", list(only_zeros_col))

df_without_store_fwdflag = df.drop(columns = ["store_and_fwd_flag","tpep_pickup_datetime","tpep_dropoff_datetime"]) # Pick all columns except store_and_fwd_flag, ,"tpep_pickup_datetime" snf ,"tpep_dropoff_datetime"

#df_without_store_fwdflag.info()

negative_values = df_without_store_fwdflag.columns[(df_without_store_fwdflag<0).any()] # columns that contain negative values
print("Columns containing negative values:", list(negative_values))

#just_fare = (df["fare_amount"]<0).any()
#print(just_fare)

Columns with only zeros: []
Columns containing negative values: ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


### 6.2 Relationship with payment type

### 6.3 Relationship with vendor

## 7. Passenger counts

From above, we have 10 variables; 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee' that contains negative values. For variables such as fare_amount, tip_amount, tolls_amaiunt, total_amount, etc, it is invalid to have negative values. Hence, this needed to be cleaned. 

In [12]:
max_passenger_count = df["passenger_count"].max()
print("The max pasenger count is:", max_passenger_count)
min_passenger_count = df["passenger_count"].min()
print("The min pasenger count is:", min_passenger_count)

The max pasenger count is: 9.0
The min pasenger count is: 0.0


The above indicates that the minimum passenger couunt (number of passengers a driver dropped on a trip) is 0 while the largest number is 9 ( These are a bit odd. A yellow taxi has 4 passenger seats, minivans/suvs can pick up to 5, and with a lap child, the absolute fleet limit is 6 hence it seems highly odd for a trip to take over 6 of such passengers. Also, my first assumption is that, a trip is counted when it picks at least a passenger from one point to the other hence making 0 a number to take a look at. Although, 0 could mean a cancelled trip. Go over the .txt file again and check for validity of these values)


In [13]:
# min function to count the number of times n number of passengers were picked per trip
def how_many_passengers_per_trip(num_of_passengers):
    print(f"The number of times {num_of_passengers} were taken per trip is", (df["passenger_count"]==num_of_passengers).sum())



In [ ]:
how_many_passengers_per_trip(9)
how_many_passengers_per_trip(8)
how_many_passengers_per_trip(7)
how_many_passengers_per_trip(6)
how_many_passengers_per_trip(0)
how_many_passengers_per_trip(1)

The number of times 9 were taken per trip is 1
The number of times 8 were taken per trip is 4
The number of times 7 were taken per trip is 2
The number of times 6 were taken per trip is 4887
The number of times 0 were taken per trip is 14787
The number of times 1 were taken per trip is 2150994


In [14]:
# You know what, let's check the frequency for this column 

# Count the number of occurence
counts_passenger = df["passenger_count"].value_counts().reset_index()

counts_passenger.columns = ['passenger_count','frequency'] # rename for clarity
print(counts_passenger)

   passenger_count  frequency
0              1.0    2150994
1              2.0     334370
2              3.0      72864
3              4.0      49738
4              0.0      14787
5              5.0       9184
6              6.0       4887
7              8.0          4
8              7.0          2
9              9.0          1


The larger count values 8, 9, 10 seem to appear as an outlier with only 4, 2, and 1 occurence respectively. As expected, the passenger count 1 is the modal value folowed by 2. An unexpected behaviour is the frequency of the passenger count 0. It occured 14,787 times. This requires further investigation.  Also, invetigate more into the NaN of the passenger count. How does this differ from passenger count equal to 0

In [15]:
check_for_NaN = df[(df["passenger_count"]).isnull()]
print(check_for_NaN.head())

         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
2636831         2  2026-01-01 00:28:47   2026-01-01 00:49:30              NaN   
2636832         2  2026-01-01 00:54:24   2026-01-01 01:15:01              NaN   
2636833         2  2026-01-01 00:12:31   2026-01-01 00:41:02              NaN   
2636834         2  2026-01-01 00:48:56   2026-01-01 01:08:00              NaN   
2636835         2  2026-01-01 00:32:09   2026-01-01 00:41:05              NaN   

         trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
2636831          10.58         NaN                NaN            43   
2636832          11.04         NaN                NaN           200   
2636833           4.25         NaN                NaN           142   
2636834           2.05         NaN                NaN            90   
2636835           0.97         NaN                NaN           237   

         DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  \
263683

## 8. Trip Duration

### 8.1 Creating duration

### 8.2 Distribution

### 8.3 Suspicious observations

## 9. Trip distance


In [ ]:
trip_dist_eq_0 = df[df["trip_distance"]==0] # trip distances that are 0

print(trip_dist_eq_0)

         VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
118             1  2026-01-01 00:21:36   2026-01-01 00:22:52              1.0   
329             1  2026-01-01 00:11:22   2026-01-01 00:34:04              1.0   
330             1  2026-01-01 00:54:54   2026-01-01 01:09:08              1.0   
360             2  2026-01-01 00:32:10   2026-01-01 00:32:14              4.0   
634             2  2026-01-01 00:45:48   2026-01-01 00:45:54              1.0   
...           ...                  ...                   ...              ...   
3723959         2  2026-01-31 23:03:31   2026-01-31 23:03:43              NaN   
3724071         2  2026-01-31 23:10:30   2026-01-31 23:16:29              NaN   
3724564         2  2026-01-31 23:37:27   2026-01-31 23:37:48              NaN   
3724651         2  2026-01-31 23:29:43   2026-01-31 23:57:04              NaN   
3724885         2  2026-01-31 23:33:53   2026-01-31 23:34:07              NaN   

         trip_distance  Rat

There were 125,738 times that the trip distance is 0. A trip distance being 0 means that the driver canceled trip and didnt go through with dropping the clients. Looking at the corresponding passenger_count column, there were times that the passenger(s) were in the car before the trip was cancelled. The most unusual thing for me here is the pickup and dropoff times.There were times that it took the driver over 30 mins to cancel the drip, recording a trip distance of 0, yet a substantial difference in the pickup and dropoff. For instance, row329, VendorID 1. What could be the reason? ALtercation between drivers and client? If so, how was this finally resolved? Something to look into. Another interesting finding is the RatecodeID. Some RatecodeIDs were 5, indicating that the final rate code in effect at the end of the trip is negotiated fare. This could mean that the reason the trip distance is 0 is because the driver and passenger couldnt agree on a fare. How do we deal with passenger counts equal to 0 and NaN? 

### Check for impossible trip

In [19]:
# Check for Trips exceeding legal limits
passenger_exceeding_6 = df[df["passenger_count"]>6]

how_many_exceeds_6 = passenger_exceeding_6["passenger_count"].sum()

print(f"There were {how_many_exceeds_6} instances where taxis picked passengers beyond 6")

There were 55.0 instances where taxis picked passengers beyond 6


This is an impossible trip because, as explained above, the maximum limit of passengers a taxi can pick is 6. Hence, any value above this should be impossible.
Now, let's check whether, of these trips above, how many of them has a trip distance of 0


In [26]:
# Check for Trips exceeding legal limits and trip distance =0
passenger_exceeding_6_and_trip_dist0 = df[(df["passenger_count"]>6) & (df["trip_distance"]==0)] 

how_many_exceeds_6_and_trip_dist0 = passenger_exceeding_6_and_trip_dist0["passenger_count"].sum()

print(f"There were {how_many_exceeds_6_and_trip_dist0} instances where taxis picked passengers beyond 6 and trip distance equal to 0")

There were 32.0 instances where taxis picked passengers beyond 6 and trip distance equal to 0


Out of the 55 instances where passengers exceeds the fleet limit, drivers cancelled the trip 32 times. W 

In [37]:
## check for destination limits (out-of-city limits). ie ratecodeid is not in the range 1-6

df_filter_ratecodeid = df[~df["RatecodeID"].isin(range(1,7))]
print("The number of ratecodeid not in the range 1-6 is:", len(df_filter_ratecodeid))
print(df_filter_ratecodeid.head())



The number of ratecodeid not in the range 1-6 is: 1198922
      VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
642          1  2026-01-01 00:11:59   2026-01-01 01:07:53              1.0   
822          1  2026-01-01 00:11:09   2026-01-01 00:47:05              1.0   
837          1  2026-01-01 00:58:46   2026-01-01 01:26:34              1.0   
1361         1  2026-01-01 00:54:38   2026-01-01 01:54:59              1.0   
1529         1  2026-01-01 00:41:52   2026-01-01 01:43:46              1.0   

      trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
642             5.2        99.0                  N            91   
822             9.3        99.0                  N            25   
837             6.3        99.0                  N            61   
1361           16.0        99.0                  N           205   
1529            7.1        99.0                  N            25   

      DOLocationID  payment_type  fare_amount  extra  mta_tax  t

## 10. Temporal Characteristics 

### 10.1 Hour

### 10.2 Day of the week

### 10.3 Weekend vs Weekday

## 11. Preliminary Findings 

## 12. Data Quality Decisions

## 13. Questions for Further Investigation